In [ ]:
import pandas as pd
import os

modelling_folder = os.environ["MODELLING_FOLDER"]
pd.set_option('max_colwidth', None)
pd.set_option('display.max_columns', None)

def single_model_performance(config, epoch):
    data = pd.read_csv(f"{modelling_folder}/metrics/{config}.csv")
    data = data[data["epoch"]==epoch]
    data = data[["dataset_subset", "f1_local_flood", "precision_local_flood", "recall_local_flood", "f1_local_no_flood", "precision_local_no_flood", "recall_local_no_flood"]]
    data = data.replace({"val_EMSR759_2024-09-21":"Hungary 2024", "val_EMSR761_2024-09-19":"Germany 2024", "val_EMSR659_2023-05-05":"Italy 2023", "val_EMSR416_2019-12-15":"France 2019",               
                        "val_EMSR692_2023-09-10":"Greece 2023", "val_EMSR757_2024-09-18":"Slovakia 2024", "val_EMSR756_2024-09-18":"Poland 2024"})
    data["dataset_subset"] = data["dataset_subset"].astype("category")
    data["dataset_subset"] = data["dataset_subset"].cat.reorder_categories(["France 2019", "Italy 2023", "Greece 2023", "Poland 2024", "Slovakia 2024", "Hungary 2024", "Germany 2024"])
    data = data.sort_values("dataset_subset").reset_index(drop=True)
    data.columns = pd.MultiIndex.from_tuples([("", "Event"), ("Flood", "F1"), ("Flood", "Precision"), ("Flood", "Recall"), ("No Flood", "F1"), ("No Flood", "Precision"), ("No Flood", "Recall")])
    return data

def multiple_model_performances(configs, epoch=100, config_names=None, sort_values=False):
    data = pd.concat([pd.read_csv(f"{modelling_folder}/metrics/{config}.csv") for config in configs])
    data = data[data["epoch"]==epoch]
    if not config_names:
        config_names = configs
    data = data.replace({config: config_name for config, config_name in zip(configs, config_names)})
    data = data.rename(columns={"config_name": "Model", "f1_local_flood": "F1", "precision_local_flood": "Precision", "recall_local_flood": "Recall"})
    data = data.replace({"val_EMSR759_2024-09-21":"Hungary 2024", "val_EMSR761_2024-09-19":"Germany 2024", "val_EMSR659_2023-05-05":"Italy 2023", "val_EMSR416_2019-12-15":"France 2019",               
                        "val_EMSR692_2023-09-10":"Greece 2023", "val_EMSR757_2024-09-18":"Slovakia 2024", "val_EMSR756_2024-09-18":"Poland 2024"})
    data = data.pivot_table(columns="dataset_subset", index="Model", values=["F1","Precision","Recall"], aggfunc="first")
    data = data[[column_name for column_name in [(metric, subset) for metric in ["F1", "Precision", "Recall"] for subset in ["France 2019", "Italy 2023", "Greece 2023", "Poland 2024", "Slovakia 2024", "Hungary 2024", "Germany 2024"]] if column_name in data.columns]]
    if sort_values:
        data = data.sort_values(sort_values, ascending=False)
    data = data.dropna(axis=1)

    display(data)

In [11]:
multiple_model_performances(["observational_evalevents_baseline", 
                             "observational_evalevents_ablation_basicnoscales",
                             "observational_evalevents_ablation_basicallscales", 
                             "observational_evalevents_ablation_nocrop",
                             "observational_evalevents_ablation_noscalesresiduals",
                            "observational_evalevents_ablation_kernel3",],
                             100,
                             ["Baseline", "No Context or Nearby Scales", "Single Encoder with all Scales", "No Cropping of Residuals", "No Residuals from Wider Scales", "Kernel Size of 3"])


F1                                     \
dataset_subset                 France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                           
Baseline                             0.805      0.505       0.789       0.650   
Kernel Size of 3                     0.791      0.413       0.762       0.594   
No Context or Nearby Scales          0.761      0.342       0.693       0.570   
No Cropping of Residuals             0.714      0.328       0.710       0.591   
No Residuals from Wider Scales       0.789      0.401       0.714       0.595   
Single Encoder with all Scales       0.554      0.313       0.708       0.579   

                                                                        \
dataset_subset                 Slovakia 2024 Hungary 2024 Germany 2024   
Model                                                                    
Baseline                               0.622        0.687        0.481   
Kernel Size of 3                       0.530        0.605        0.485   
No Context or Nearby Scales            0.558        0.599        0.427   
No Cropping of Residuals               0.559        0.571        0.415   
No Residuals from Wider Scales         0.543        0.607        0.444   
Single Encoder with all Scales         0.544        0.520        0.413   

                                 Precision                                     \
dataset_subset                 France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                           
Baseline                             0.716      0.575       0.757       0.689   
Kernel Size of 3                     0.708      0.476       0.691       0.541   
No Context or Nearby Scales          0.640      0.344       0.578       0.508   
No Cropping of Residuals             0.565      0.366       0.595       0.563   
No Residuals from Wider Scales       0.725      0.333       0.603       0.600   
Single Encoder with all Scales       0.385      0.266       0.607       0.557   

                                                                        \
dataset_subset                 Slovakia 2024 Hungary 2024 Germany 2024   
Model                                                                    
Baseline                               0.614        0.661        0.455   
Kernel Size of 3                       0.485        0.522        0.399   
No Context or Nearby Scales            0.523        0.499        0.330   
No Cropping of Residuals               0.508        0.534        0.305   
No Residuals from Wider Scales         0.508        0.611        0.387   
Single Encoder with all Scales         0.562        0.471        0.330   

                                    Recall                                     \
dataset_subset                 France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                           
Baseline                             0.919      0.451       0.825       0.616   
Kernel Size of 3                     0.895      0.365       0.850       0.659   
No Context or Nearby Scales          0.938      0.339       0.866       0.650   
No Cropping of Residuals             0.968      0.296       0.878       0.623   
No Residuals from Wider Scales       0.865      0.505       0.874       0.590   
Single Encoder with all Scales       0.984      0.382       0.850       0.602   

                                                                        
dataset_subset                 Slovakia 2024 Hungary 2024 Germany 2024  
Model                                                                   
Baseline                               0.629        0.714        0.509  
Kernel Size of 3                       0.584        0.720        0.618  
No Context or Nearby Scales            0.599        0.748        0.603  
No Cropping of Residuals               0.621        0.612        0.647  
No Re

In [13]:
multiple_model_performances(["observational_evalevents_baseline", "observational_finetune_checker_good"], 
                             config_names=["Baseline", "Pre-trained and Fine-tuned"])

F1                                     \
dataset_subset             France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                       
Baseline                         0.805      0.505       0.789       0.650   
Pre-trained and Fine-tuned       0.833      0.541       0.790       0.656   

                                                                    \
dataset_subset             Slovakia 2024 Hungary 2024 Germany 2024   
Model                                                                
Baseline                           0.622        0.687        0.481   
Pre-trained and Fine-tuned         0.609        0.692        0.428   

                             Precision                                     \
dataset_subset             France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                       
Baseline                         0.716      0.575       0.757       0.689   
Pre-trained and Fine-tuned       0.806      0.615       0.758       0.705   

                                                                    \
dataset_subset             Slovakia 2024 Hungary 2024 Germany 2024   
Model                                                                
Baseline                           0.614        0.661        0.455   
Pre-trained and Fine-tuned         0.565        0.648        0.337   

                                Recall                                     \
dataset_subset             France 2019 Italy 2023 Greece 2023 Poland 2024   
Model                                                                       
Baseline                         0.919      0.451       0.825       0.616   
Pre-trained and Fine-tuned       0.862      0.482       0.825       0.614   

                                                                    
dataset_subset             Slovakia 2024 Hungary 2024 Germany 2024  
Model                                                               
Baseline                           0.629        0.714        0.509  
Pre-trained and Fine-tuned         0.661        0.742        0.588

In [21]:
multiple_model_performances(["observational_pretrained_all_overlap_side_split", "observational_evalevents_comparison_split"], 100,
                            ["Spatially Overlapping Events and Regional Split", "Other Countries' Events and Regional Split"])

F1             \
dataset_subset                                  France 2019 Italy 2023   
Model                                                                    
Other Countries' Events and Regional Split            0.726      0.004   
Spatially Overlapping Events and Regional Split       0.741      0.265   

                                                              Precision  \
dataset_subset                                  Greece 2023 France 2019   
Model                                                                     
Other Countries' Events and Regional Split            0.466       0.864   
Spatially Overlapping Events and Regional Split       0.695       0.908   

                                                                        \
dataset_subset                                  Italy 2023 Greece 2023   
Model                                                                    
Other Countries' Events and Regional Split            0.02       0.351   
Spatially Overlapping Events and Regional Split       0.35       0.731   

                                                     Recall             \
dataset_subset                                  France 2019 Italy 2023   
Model                                                                    
Other Countries' Events and Regional Split            0.627      0.002   
Spatially Overlapping Events and Regional Split       0.626      0.214   

                                                             
dataset_subset                                  Greece 2023  
Model                                                        
Other Countries' Events and Regional Split            0.695  
Spatially Overlapping Events and Regional Split       0.663

In [19]:
multiple_model_performances(["observational_evalevents_comparison_nofrance",
                             "observational_overlap_select_france_subevent"], sort_values=("F1", "France 2019"),
                             config_names=["Other Countries' Events", "Spatially Overlapping Events"])
multiple_model_performances(["observational_evalevents_comparison_nogreece",
                             "observational_overlap_all_greece_subevent_coords"], sort_values=("F1", "Greece 2023"),
                             config_names=["Other Countries' Events", "Spatially Overlapping Events"])
multiple_model_performances(["observational_evalevents_comparison_noitaly",
                             "observational_overlap_all_italy_subevent",], sort_values=("F1", "Italy 2023"),
                             config_names=["Other Countries' Events", "Spatially Overlapping Events"]);

,F1,Precision,Recall
dataset_subset,France 2019,France 2019,France 2019
Model,,,
Spatially Overlapping Events,0.343,0.539,0.252
Other Countries' Events,0.244,0.147,0.704


,F1,Precision,Recall
dataset_subset,Greece 2023,Greece 2023,Greece 2023
Model,,,
Other Countries' Events,0.399,0.304,0.582
Spatially Overlapping Events,0.136,0.561,0.078


,F1,Precision,Recall
dataset_subset,Italy 2023,Italy 2023,Italy 2023
Model,,,
Spatially Overlapping Events,0.296,0.452,0.22
Other Countries' Events,0.191,0.106,0.98
